<a href="https://colab.research.google.com/github/sharniks/langchain-fundamentals/blob/main/agents_in_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [46]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN') # Make sure you've added your HF_TOKEN to Colab secrets!
os.environ["WEATHER_API_KEY"] = userdata.get('WEATHER_API_KEY') # Add your Weatherstack API key to Colab secrets!

In [34]:
!pip install -q langchain-huggingface langchain-community langchain-core requests duckduckgo-search ddgs

In [35]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.tools import tool
import requests

In [36]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

results = search_tool.invoke('what is the news in india')

In [37]:
results

'2 hours ago - A resurfaced Barcelona photograph and a clash of styles have given the match an added sense of history. ... The monsoon is set for a strong comeback early next week, with widespread rain expected over North, East and Central India. 19 hours ago - Gold rose on Friday as the dollar weakened and expectations of U.S. interest rate hikes eased slightly following inflation data, though prices were still on track for a fourth consecutive weekly decline. ... India\'s central bank on Thursday issued final rules for a proposed expansion of the country\'s credit derivative market, after the federal finance minister proposed deepening it in this year\'s budget. 7 hours ago - BBC\'s Azadeh Moshiri reports as thousands attempt to march to India\'s parliament in support of the Cockroach Janta Party\'s demand for education reforms. ... Why hunger strikes still shape Indian politics, from Gandhi to Sonam Wangchuk. ... The World Cricketers\' Association says it is "concerned" by the chang

In [48]:
WEATHER_API_KEY = os.environ["WEATHER_API_KEY"]

@tool
def get_weather_data(city:str):
  """
  This function fetches the current weather data for a given city
  """
  url = f'https://api.weatherstack.com/current?access_key={WEATHER_API_KEY}&query={city}'

  response = requests.get(url)

  return response.json()

In [39]:
llm_endpoint = HuggingFaceEndpoint(
    repo_id='Qwen/Qwen2.5-72B-Instruct',
    task='text-generation'
)

llm = ChatHuggingFace(llm=llm_endpoint)

In [41]:
from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_classic import hub
from langchain_core.prompts import PromptTemplate

In [42]:
# Step 2: Pull the ReAct prompt from LangChain Hub //not working now
template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate.from_template(template)

In [43]:
# Step 3: Create the ReAct agent manually with the pulled prompt
agent = create_react_agent(
    llm=llm,
    tools=[search_tool, get_weather_data],
    prompt=prompt
)

In [44]:
# Step 4: Wrap it with AgentExecutor
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool, get_weather_data],
    verbose=True
)

In [49]:
# Step 5: Invoke
response = agent_executor.invoke({"input": "Find the capital of Madhya Pradesh, then find it's current weather condition"})
print(response)



> Entering new AgentExecutor chain...
I need to find the capital of Madhya Pradesh first.
Action: duckduckgo_search
Action Input: Capital of Madhya Pradesh
Observ2 days ago - This is considered as a remarkable and significant milestone putting Madhya Pradesh as music tourism hub worldwide. Dainik Bhaskar, Dainik Jagran, The Indian Observer, Nava Bharat, Deshbandhu, Nai Duniya, Rajasthan Patrika, Raj Express and Dainik Dabang Dunia are the leading Hindi newspapers. June 13, 2026 - Bhopal also has many other mega projects lined up in its vicinity. In March 2022, Madhya Pradesh government announced the development of Bagroda Industrial Area Phase-2 after observing the immense interest of investors to set up manufacturing units near Bhopal. 1 week ago - Madhya Pradesh, meaning "central state," appropriately sits in the heart of India, without any coasts or international borders. Bhopal is its capital. April 15, 2026 - Its status as the largest city in Madhya Pradesh by population makes i

In [50]:
response['output']

'The capital of Madhya Pradesh is Bhopal. The current weather condition in Bhopal is partly cloudy with a temperature of 28°C (feels like 33°C), humidity of 74%, and a wind speed of 4 km/h from the north. The visibility is 6 km, and the UV index is 1.'